# 📊 ANÁLISIS COMPARATIVO DE MODELOS - CALIDAD DE CAFÉ

## 🎯 Objetivo
Analizar y comparar el rendimiento de diferentes modelos de Machine Learning entrenados para predecir la calidad del café.

## 📋 Contenido
1. Carga de resultados generados por el pipeline
2. Análisis de métricas de rendimiento
3. Comparación visual de modelos
4. Análisis de curva ROC
5. Selección del mejor modelo
6. Conclusiones y recomendaciones

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('default')  # Cambiado para evitar errores
sns.set_palette("husl")

print("📚 Librerías importadas correctamente")
print(f"📅 Fecha de análisis: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Cargar resultados del entrenamiento
try:
    # Rutas corregidas para Windows
    base_path = os.path.dirname(os.getcwd())  # Subir un nivel desde notebooks
    results_path = os.path.join(base_path, 'models', 'prediction', 'training_results.csv')
    roc_path = os.path.join(base_path, 'models', 'prediction', 'roc_data.pkl')
    metadata_path = os.path.join(base_path, 'models', 'prediction', 'training_metadata.pkl')
    
    print(f"📁 Buscando archivos en: {base_path}")
    print(f"📊 Results: {results_path}")
    
    # Cargar resultados de entrenamiento
    results_df = pd.read_csv(results_path)
    
    # Cargar datos ROC
    roc_data = joblib.load(roc_path)
    
    # Cargar metadatos
    metadata = joblib.load(metadata_path)
    
    print("✅ Datos cargados exitosamente")
    print(f"📊 Resultados de {len(results_df)} modelos")
    print(f"🥇 Mejor modelo: {metadata['best_model_name']}")
    print(f"📊 RMSE mejor: {metadata['best_rmse']:.3f}")
    print(f"📊 R² mejor: {metadata['best_r2']:.3f}")
    
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("💡 Asegúrate de haber ejecutado primero el pipeline de entrenamiento")
    print("📁 Verifica que los archivos existan en models/prediction/")
    
    # Listar archivos que sí existen
    models_dir = os.path.join(base_path, 'models', 'prediction')
    if os.path.exists(models_dir):
        print(f"📁 Archivos encontrados en {models_dir}:")
        for file in os.listdir(models_dir):
            print(f"   - {file}")
    else:
        print(f"❌ Directorio {models_dir} no existe")
        
except Exception as e:
    print(f"❌ Error inesperado: {e}")

## 📈 1. Análisis General de Resultados

In [ ]:
# Tabla completa de resultados
print("📊 TABLA COMPLETA DE RESULTADOS")
print("="*80)

# Verificar que tenemos datos
if 'results_df' in locals():
    # Seleccionar columnas relevantes
    display_columns = ['model_name', 'test_rmse', 'test_r2', 'test_mae', 'cv_rmse_mean', 'training_time']
    
    # Verificar que las columnas existan
    available_columns = [col for col in display_columns if col in results_df.columns]
    print(f"📋 Columnas disponibles: {available_columns}")
    
    results_display = results_df[available_columns].copy()
    
    # Formatear para mejor visualización
    if 'test_rmse' in results_display.columns:
        results_display['test_rmse'] = results_display['test_rmse'].round(3)
    if 'test_r2' in results_display.columns:
        results_display['test_r2'] = results_display['test_r2'].round(3)
    if 'test_mae' in results_display.columns:
        results_display['test_mae'] = results_display['test_mae'].round(3)
    if 'cv_rmse_mean' in results_display.columns:
        results_display['cv_rmse_mean'] = results_display['cv_rmse_mean'].round(3)
    if 'training_time' in results_display.columns:
        results_display['training_time'] = results_display['training_time'].round(2)
    
    # Renombrar columnas
    column_mapping = {
        'model_name': 'Modelo',
        'test_rmse': 'RMSE Test',
        'test_r2': 'R² Test',
        'test_mae': 'MAE Test',
        'cv_rmse_mean': 'RMSE CV',
        'training_time': 'Tiempo (s)'
    }
    
    results_display.columns = [column_mapping.get(col, col) for col in results_display.columns]
    
    # Ordenar por RMSE si existe
    if 'RMSE Test' in results_display.columns:
        results_display = results_display.sort_values('RMSE Test')
    
    # Mostrar tabla
    display(results_display)
    
else:
    print("❌ No se cargaron los resultados. Revisa la celda anterior.")

In [ ]:
# Estadísticas descriptivas de las métricas
if 'results_df' in locals():
    print("📈 ESTADÍSTICAS DESCRIPTIVAS DE MÉTRICAS")
    print("="*50)
    
    metrics = ['test_rmse', 'test_r2', 'test_mae', 'cv_rmse_mean']
    metric_names = ['RMSE Test', 'R² Test', 'MAE Test', 'RMSE CV']
    
    stats_summary = []
    for metric, name in zip(metrics, metric_names):
        if metric in results_df.columns:
            stats = {
                'Métrica': name,
                'Mínimo': results_df[metric].min(),
                'Máximo': results_df[metric].max(),
                'Promedio': results_df[metric].mean(),
                'Desviación': results_df[metric].std()
            }
            stats_summary.append(stats)
    
    if stats_summary:
        stats_df = pd.DataFrame(stats_summary)
        display(stats_df.round(3))
    else:
        print("⚠️ No se encontraron métricas para analizar")
else:
    print("❌ No se cargaron los resultados. Revisa las celdas anteriores.")

## 📊 2. Visualizaciones Comparativas

In [ ]:
# Gráfico comparativo principal
if 'results_df' in locals():
    try:
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Análisis Comparativo de Modelos - Calidad de Café', fontsize=16, fontweight='bold')
        
        # 1. RMSE Test (Error)
        if 'test_rmse' in results_df.columns:
            results_sorted_rmse = results_df.sort_values('test_rmse')
            bars1 = ax1.barh(range(len(results_sorted_rmse)), results_sorted_rmse['test_rmse'], 
                             color='#2E86AB', alpha=0.8)
            ax1.set_xlabel('RMSE Test (menor es mejor)', fontweight='bold')
            ax1.set_title('Error Cuadrático Medio', fontweight='bold')
            ax1.set_yticks(range(len(results_sorted_rmse)))
            ax1.set_yticklabels(results_sorted_rmse['model_name'], fontsize=9)
            ax1.grid(axis='x', alpha=0.3)
            
            # Resaltar mejor modelo
            bars1[0].set_color('#F18F01')
            bars1[0].set_alpha(1.0)
        
        # 2. R² Test (Precisión)
        if 'test_r2' in results_df.columns:
            results_sorted_r2 = results_df.sort_values('test_r2', ascending=True)
            bars2 = ax2.barh(range(len(results_sorted_r2)), results_sorted_r2['test_r2'], 
                             color='#A23B72', alpha=0.8)
            ax2.set_xlabel('R² Test (mayor es mejor)', fontweight='bold')
            ax2.set_title('Coeficiente de Determinación', fontweight='bold')
            ax2.set_yticks(range(len(results_sorted_r2)))
            ax2.set_yticklabels(results_sorted_r2['model_name'], fontsize=9)
            ax2.grid(axis='x', alpha=0.3)
            
            # Resaltar mejor modelo
            best_idx2 = results_sorted_r2['test_r2'].idxmax()
            bars2[best_idx2].set_color('#F18F01')
            bars2[best_idx2].set_alpha(1.0)
        
        # 3. Tiempo de entrenamiento
        if 'training_time' in results_df.columns:
            results_sorted_time = results_df.sort_values('training_time', ascending=True)
            bars3 = ax3.barh(range(len(results_sorted_time)), results_sorted_time['training_time'], 
                             color='#592E83', alpha=0.8)
            ax3.set_xlabel('Tiempo (segundos)', fontweight='bold')
            ax3.set_title('Tiempo de Entrenamiento', fontweight='bold')
            ax3.set_yticks(range(len(results_sorted_time)))
            ax3.set_yticklabels(results_sorted_time['model_name'], fontsize=9)
            ax3.grid(axis='x', alpha=0.3)
        
        # 4. RMSE vs R² (Scatter)
        if 'test_rmse' in results_df.columns and 'test_r2' in results_df.columns:
            scatter = ax4.scatter(results_df['test_rmse'], results_df['test_r2'], 
                                s=100, alpha=0.7, c=range(len(results_df)), cmap='viridis')
            ax4.set_xlabel('RMSE Test', fontweight='bold')
            ax4.set_ylabel('R² Test', fontweight='bold')
            ax4.set_title('RMSE vs R²', fontweight='bold')
            ax4.grid(True, alpha=0.3)
            
            # Añadir etiquetas
            for i, (_, row) in enumerate(results_df.iterrows()):
                ax4.annotate(row['model_name'].split()[0], (row['test_rmse'], row['test_r2']),
                            xytext=(3, 3), textcoords='offset points', fontsize=8)
            
            # Líneas de referencia
            ax4.axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='R² = 0.7')
            ax4.axvline(x=2.0, color='red', linestyle='--', alpha=0.5, label='RMSE = 2.0')
            ax4.legend()
        
        plt.tight_layout()
        plt.show()
        
        print("📊 Visualizaciones generadas")
        
    except Exception as e:
        print(f"❌ Error al generar gráficos: {e}")
else:
    print("❌ No se cargaron los resultados. Revisa las celdas anteriores.")

## 📈 3. Análisis de Curva ROC

In [ ]:
# Visualización de Curvas ROC
if 'roc_data' in locals():
    try:
        plt.figure(figsize=(12, 8))
        
        # Línea base
        plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Clasificador Aleatorio')
        
        # Colores para diferentes modelos
        colors = plt.cm.Set3(np.linspace(0, 1, len(roc_data)))
        
        for i, (model_name, roc_info) in enumerate(roc_data.items()):
            plt.plot(roc_info['fpr'], roc_info['tpr'], 
                    color=colors[i], linewidth=2,
                    label=f'{model_name} (AUC = {roc_info["auc"]:.3f})')
        
        plt.xlabel('False Positive Rate', fontweight='bold', fontsize=12)
        plt.ylabel('True Positive Rate', fontweight='bold', fontsize=12)
        plt.title('Curva ROC - Comparación de Modelos', fontweight='bold', fontsize=14)
        plt.legend(loc='lower right', fontsize=9)
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Tabla de AUC
        auc_data = []
        for model_name, roc_info in roc_data.items():
            auc_data.append({
                'Modelo': model_name,
                'AUC': roc_info['auc']
            })
        
        auc_df = pd.DataFrame(auc_data).sort_values('AUC', ascending=False)
        print("📊 TABLA DE AUC (ÁREA BAJO LA CURVA ROC)")
        print("="*50)
        display(auc_df.round(3))
        
    except Exception as e:
        print(f"❌ Error al generar curvas ROC: {e}")
else:
    print("❌ No se cargaron los datos ROC. Revisa las celdas anteriores.")

## 🏆 4. Análisis del Mejor Modelo

In [ ]:
# Información detallada del mejor modelo
if 'metadata' in locals() and 'results_df' in locals():
    try:
        best_model_name = metadata['best_model_name']
        best_results = results_df[results_df['model_name'] == best_model_name].iloc[0]
        
        print(f"🏆 ANÁLISIS DETALLADO - MEJOR MODELO: {best_model_name}")
        print("="*60)
        
        print(f"📊 Métricas de rendimiento:")
        if 'test_rmse' in best_results:
            print(f"   - RMSE Test: {best_results['test_rmse']:.3f}")
        if 'test_r2' in best_results:
            print(f"   - R² Test: {best_results['test_r2']:.3f}")
        if 'test_mae' in best_results:
            print(f"   - MAE Test: {best_results['test_mae']:.3f}")
        if 'cv_rmse_mean' in best_results:
            print(f"   - RMSE CV: {best_results['cv_rmse_mean']:.3f} ± {best_results['cv_rmse_std']:.3f}")
        if 'training_time' in best_results:
            print(f"   - Tiempo entrenamiento: {best_results['training_time']:.2f} segundos")
        
        # Interpretación del rendimiento
        if 'test_rmse' in best_results and 'test_r2' in best_results:
            rmse = best_results['test_rmse']
            r2 = best_results['test_r2']
            
            print(f"\n🎯 Interpretación práctica:")
            print(f"   - Error promedio: ±{rmse:.2f} puntos en escala 0-100")
            print(f"   - Varianza explicada: {r2*100:.1f}% de la variabilidad")
            
            if r2 > 0.7:
                print(f"   ✅ Modelo con buen poder predictivo (R² > 0.7)")
            elif r2 > 0.5:
                print(f"   ⚠️ Modelo con poder predictivo moderado (0.5 < R² < 0.7)")
            else:
                print(f"   ❌ Modelo con bajo poder predictivo (R² < 0.5)")
            
            if rmse < 2.0:
                print(f"   ✅ Error aceptable para catación profesional (< 2.0 puntos)")
            else:
                print(f"   ⚠️ Error elevado para aplicación práctica (> 2.0 puntos)")
        
    except Exception as e:
        print(f"❌ Error al analizar mejor modelo: {e}")
else:
    print("❌ No se cargaron los metadatos o resultados. Revisa las celdas anteriores.")

In [ ]:
# Cargar y analizar el mejor modelo
try:
    base_path = os.path.dirname(os.getcwd())
    model_path = os.path.join(base_path, 'models', 'prediction', 'best_model.pkl')
    preprocessor_path = os.path.join(base_path, 'models', 'prediction', 'preprocessor.pkl')
    
    best_model = joblib.load(model_path)
    preprocessor = joblib.load(preprocessor_path)
    
    print(f"🤖 INFORMACIÓN DEL MODELO:")
    print(f"   - Tipo: {type(best_model).__name__}")
    print(f"   - Parámetros: {best_model.get_params()}")
    
    # Si tiene feature importances
    if hasattr(best_model, 'feature_importances_'):
        print(f"\n📊 IMPORTANCIA DE VARIABLES:")
        
        # Obtener nombres de features del preprocesador
        feature_names = preprocessor.get_feature_names_out()
        
        # Crear DataFrame con importancias
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': best_model.feature_importances_
        })
        
        # Top 15 features más importantes
        top_features = importance_df.sort_values('importance', ascending=False).head(15)
        
        print(f"🎯 Top 15 variables más importantes:")
        for i, (_, row) in enumerate(top_features.iterrows(), 1):
            print(f"   {i:2d}. {row['feature']}: {row['importance']:.4f}")
        
        # Visualización
        plt.figure(figsize=(12, 8))
        bars = plt.barh(range(len(top_features)), top_features['importance'], 
                        color='#2E86AB', alpha=0.8)
        plt.yticks(range(len(top_features)), top_features['feature'])
        plt.xlabel('Importancia', fontweight='bold')
        plt.title('Importancia de Variables - Mejor Modelo', fontweight='bold', fontsize=14)
        plt.gca().invert_yaxis()
        plt.grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
except Exception as e:
    print(f"❌ Error al cargar el modelo: {e}")

## 🎯 5. Conclusiones y Recomendaciones

In [ ]:
# Resumen de conclusiones
if 'metadata' in locals() and 'results_df' in locals():
    try:
        print("🎯 CONCLUSIONES Y RECOMENDACIONES")
        print("="*60)
        
        # 1. Mejor modelo
        best_model_name = metadata['best_model_name']
        best_results = results_df[results_df['model_name'] == best_model_name].iloc[0]
        
        print(f"\n🏆 MEJOR MODELO SELECCIONADO:")
        print(f"   - Modelo: {best_model_name}")
        if 'test_rmse' in best_results:
            print(f"   - RMSE: {best_results['test_rmse']:.3f} puntos")
        if 'test_r2' in best_results:
            print(f"   - R²: {best_results['test_r2']:.3f} ({best_results['test_r2']*100:.1f}% varianza explicada)")
        if 'training_time' in best_results:
            print(f"   - Tiempo de entrenamiento: {best_results['training_time']:.2f} segundos")
        
        # 2. Ranking de modelos
        print(f"\n📊 RANKING DE MODELOS (Top 5):")
        if 'test_rmse' in results_df.columns:
            top_5 = results_df.nsmallest(5, 'test_rmse')
            for i, (_, row) in enumerate(top_5.iterrows(), 1):
                status = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}"
                rmse_val = f"RMSE={row['test_rmse']:.3f}" if 'test_rmse' in row else "RMSE=N/A"
                r2_val = f"R²={row['test_r2']:.3f}" if 'test_r2' in row else "R²=N/A"
                print(f"   {status} {row['model_name']}: {rmse_val}, {r2_val}")
        
        # 3. Análisis de rendimiento
        print(f"\n📈 ANÁLISIS DE RENDIMIENTO:")
        if 'test_rmse' in results_df.columns:
            avg_rmse = results_df['test_rmse'].mean()
            std_rmse = results_df['test_rmse'].std()
            print(f"   - RMSE promedio: {avg_rmse:.3f} ± {std_rmse:.3f}")
            print(f"   - Variabilidad en RMSE: {(std_rmse/avg_rmse)*100:.1f}%")
        
        if 'test_r2' in results_df.columns:
            avg_r2 = results_df['test_r2'].mean()
            print(f"   - R² promedio: {avg_r2:.3f}")
        
        # 4. Recomendaciones
        print(f"\n💡 RECOMENDACIONES:")
        
        if 'test_rmse' in best_results:
            rmse = best_results['test_rmse']
            if rmse < 1.5:
                print(f"   ✅ EXCELENTE: El mejor modelo tiene error < 1.5 puntos")
                print(f"      → Adecuado para producción y catación profesional")
            elif rmse < 2.0:
                print(f"   ✅ BUENO: El mejor modelo tiene error < 2.0 puntos")
                print(f"      → Adecuado para aplicaciones prácticas")
            else:
                print(f"   ⚠️ REGULAR: El mejor modelo tiene error > 2.0 puntos")
                print(f"      → Requiere mejora antes de producción")
        
        if 'test_r2' in best_results:
            r2 = best_results['test_r2']
            if r2 > 0.8:
                print(f"   ✅ EXCELENTE: El modelo explica > 80% de la variabilidad")
            elif r2 > 0.7:
                print(f"   ✅ BUENO: El modelo explica > 70% de la variabilidad")
            else:
                print(f"   ⚠️ REGULAR: El modelo explica < 70% de la variabilidad")
        
        # 5. Próximos pasos
        print(f"\n🚀 PRÓXIMOS PASOS SUGERIDOS:")
        print(f"   1. 📊 Implementar el mejor modelo en producción")
        print(f"   2. 🔧 Usar demo_prediction.py para predicciones")
        print(f"   3. 📈 Monitorear rendimiento con datos reales")
        print(f"   4. 🧪 Experimentar con ensemble de mejores modelos")
        print(f"   5. 📊 Desarrollar dashboard de predicciones")
        
        print(f"\n✅ Análisis completado exitosamente!")
        print(f"📅 Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
    except Exception as e:
        print(f"❌ Error en conclusiones: {e}")
else:
    print("❌ No se cargaron los datos necesarios. Revisa las celdas anteriores.")